## Patch-seq Benchmarks

These models are all trained on the full Patch-seq dataset (though a given model may not have use for every sample), using a 120 x 4 x 4 arbor density.

### Random Forest "Auto-encoder"

The random forest AE model is trained to predict the T-type label of a sample at a certain level of the hierarchy, from which it will generate a reconstruction equal to the mean of training samples within that cluster.

The model labeling scheme is "raw_{merge number}", where the "merge number" refers to how many clusters have been combined at the hierarchy level learned by that classifier.

![](summary_figures/trees.png)

### End-to-End Models

These models are variants of the VAE architecture that have a 100-dimensional "latent space" and are optimized only to minimize the cross-modal MSE.

![](summary_figures/endtoend.png)

### Variational Auto-encoder

These models map data to and from a 3-dimensional latent space using neural networks (dense for T and E, convolutional for M). They are optimized using a variational loss function that includes both cross-modal MSE terms and an explicit latent-space coupling penalty.

![](summary_figures/vae_baselines.png)

## Incorporation of Morphology Modality

### Does adding morphology improve the other modalities?

Comparison of the within-modality and cross-modality reconstruction performance for coupled-VAE models trained on Patch-seq data. Looking for evidence of improvement for the transcriptomic and electro-physiological modalities. Don't really see any.

![](summary_figures/adding_m.png)

### Coupled-VAE approach versus benchmarks

Comparison of reconstructions for morphological data (either and input or output) from the coupled-VAE model versus pairwise models trained for direct inference between modalities. M->E seems to be significantly better in the VAE model than either direct regression or random forest.

![](summary_figures/morph_regression.png)

### Different arbor density featurizations

This table compares the performance of different arbor density shapes, alongside handcrafted features (labeled "ivscc") generated by the [neuron_morphology](https://github.com/AllenInstitute/neuron_morphology) package. The last three "X->M" columns should be compared with caution, since the reconstructed data structure is changing between each row.

Note that the model label scheme is "{depth bins}\_{radial bins}\_{type bins}", with the "120_4_1" row having combined all arbor types into a single histogram. The bottom six rows also have final "_{scale}" string in their title, which indicates what value the x,y coordinates were divided by before rescaling. So the "120_4_4_100" histograms had the SWC coodinates divided by 100. All of the other rows were scaled by a value of 1000.

Based on these results, the shape of the arbor densities seems to have relatively little effect on the performance of the model outside of extreme edge cases. That the 120x4x1 arbor densities perform as well as the 120x120x4 densities strongly suggests the model is ignoring all radial information. This is reinfored by the fact that the histograms with much more relaxed scaling (e.g 120_4_4_100 scaled by 1/100) don't do any better (and in fact do somewhat worse) than densities scaled by 1/1000. The only two cases where a notable drop in performance occurs are the 120x4x1 and 1x1x4 trials. These trials lack information about arbor type and cortical depth respectively, and do significantly worse. That at least indicates those two features are important to the model. Finally, the handcrafted neuron morphology features match or outperform the arbor densities across all relevant tasks (comparison for X->M tasks must be done with great care).

![](summary_figures/all_arbors.png)

## Incorporating Partial-modality Data

This table compares the performance (on Patch-seq) of models trained on tiple-modality Patch-seq samples (first row), all Patch-seq samples (second row), Patch-seq plus ~40,000 EM morphology-only samples (third row), and Patch-seq plus EM plus ~10,000 Smart-seq transcriptomics-only samples (last row). Inclusion of partial Patch-seq samples is universally beneficial, while the addition of EM samples never really hurts the model (aside from M->M performance) and helps in E->M and T->M. Adding Smart-seq samples has a more muted effect, but may also help E->M.

![](summary_figures/partial_samples.png)

## Loss functions

This table compares the Patch-seq of performance of models trained using different loss functions. The first row is the standard variational loss function plus a small explicit coupling term between modalities, while the second row does not include this coupling term (Approach 3 in notes). The third row uses the non-variational loss function from Rohan's paper without any explicit coupling, while the fourth row has explicit coupling but no cross-modal reconstruction (using both gives results similar to the third row). The fifth row uses Approach 1 from the notes.

Based on these results it appears that the explicit coupling term provide only a very mild benefit (though doesn't seem to hurt). Conversly, the cross-modal terms in the loss are critical to achieving higher reconstruction scores, with coupling along being inadequate for cross-modal mapping involving M (especially with E). The model based on Approach does a bit better in this regard (which is interesting), but still falls short of the standard variational and non-variational models. 

![](summary_figures/loss.png)